In [ ]:
#@title Install Java version 17 due to Neo4j requirements
!apt-get install openjdk-17-jre-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java
!java -version

openjdk version "17.0.12" 2024-07-16
OpenJDK Runtime Environment (build 17.0.12+7-Ubuntu-1ubuntu222.04)
OpenJDK 64-Bit Server VM (build 17.0.12+7-Ubuntu-1ubuntu222.04, mixed mode, sharing)


In [ ]:
#@title Installing Neo4J server on Google colab
!wget -q https://neo4j.com/artifact.php?name=neo4j-community-5.21.0-unix.tar.gz -O neo4j.tar.gz
# decompress and rename
!tar -xf neo4j.tar.gz  # or --strip-components=1
!mv neo4j-community-5.21.0 nj
!wget -q https://github.com/neo4j/apoc/releases/download/5.21.0/apoc-5.21.0-core.jar -O nj/plugins/apoc-5.21.0-core.jar
# Allow apoc command
!echo "dbms.security.procedures.unrestricted=apoc.*" >> nj/conf/neo4j.conf
!echo "dbms.security.procedures.allowlist=apoc.*" >> nj/conf/neo4j.conf
# disable password, and start server
!sed -i '/#dbms.security.auth_enabled/s/^#//g' nj/conf/neo4j.conf
# start Neo4j server
!nj/bin/neo4j start

Directories in use:
home:         /content/nj
config:       /content/nj/conf
logs:         /content/nj/logs
plugins:      /content/nj/plugins
import:       /content/nj/import
data:         /content/nj/data
certificates: /content/nj/certificates
licenses:     /content/nj/licenses
run:          /content/nj/run
Starting Neo4j.
Started neo4j (pid:885). It is available at http://localhost:7474
There may be a short delay until the server is ready.


In [ ]:
#@title Install python plug to connect to Neo4j server
!pip install py2neo -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.2/177.2 kB 4.0 MB/s eta 0:00:00


In [ ]:
from py2neo import Graph
graph = Graph("bolt://localhost:7687")
graph

Graph('bolt://localhost:7687')

In [ ]:
#@title Create a jupyter command to run cypher queries
from IPython.core.magic import register_cell_magic
from IPython.core.magic_arguments import argument, magic_arguments, parse_argstring
@magic_arguments()
@argument('-df', action='store_true',help='enable pandas dataframe')
@register_cell_magic
def neo4j(line, cell=None):
    line = line.split('#')[0] if '#' in line else line
    args = parse_argstring(neo4j, line)
    result = graph.run(cell or line)
    return result.to_data_frame() if args.df else result.to_table()

Cypher Queries

In [ ]:
%%neo4j
MATCH (n)
WHERE n:Movie OR n:Person
DETACH DELETE n

In [ ]:
%%neo4j -df
MATCH (p)
RETURN p

""


In [ ]:
%%neo4j # Creat Alice and Bob Person entities
CREATE (a:Person {name: 'Alice', age: 1})
CREATE (b:Person {name: 'Bob', age: 2})

In [ ]:
%%neo4j # Create a relationship between Alice and Bob
MATCH (a:Person {name: 'Alice'}), (b:Person {name: 'Bob'})
CREATE (a)-[:KNOWS]->(b)

In [ ]:
%%neo4j # Query to find Alice Person
MATCH (a:Person {name: 'Alice'})
RETURN a,id(a)

a,id(a)
"(_0:Person {age: 1, name: 'Alice'})",0


In [ ]:
%%neo4j # Query to find Bob Person
MATCH (a:Person {name: 'Bob'})
RETURN a,id(a)

a,id(a)
"(_1:Person {age: 2, name: 'Bob'})",1


In [ ]:
%%neo4j # Query to find relationships of Alice Person
MATCH (a:Person {name: 'Alice'})-[r:KNOWS]->(b)
RETURN a, r, b

a,r,b
"(_0:Person {age: 1, name: 'Alice'})",(Alice)-[:KNOWS {}]->(Bob),"(_1:Person {age: 2, name: 'Bob'})"


In [ ]:
%%neo4j -df# Query to find relationships of Alice Person
MATCH (a:Person {name: 'Alice'})-[r:KNOWS]->(b)
RETURN a, r, b, id(r)

,a,r,b,id(r)
0,"{'name': 'Alice', 'age': 1}",{},"{'name': 'Bob', 'age': 2}",0


In [ ]:
%%neo4j # Create a Person and a Movie
CREATE (p:Person {name: 'Charlie', age: 35, city: 'New York'})
CREATE (m:Movie {title: 'Inception', release_year: 2010})

In [ ]:
%%neo4j # Create relationships between Person and Movie, Relation type: WATCHED
MATCH (p:Person {name: 'Charlie'}), (m:Movie {title: 'Inception'})
CREATE (p)-[:WATCHED {date: '2023-01-01'}]->(m)

In [ ]:
%%neo4j
CREATE (m:Movie {title: 'interstellar', release_year: 2014})

In [ ]:
%%neo4j
MATCH (p:Person {name: 'Charlie'}), (m:Movie {title: 'interstellar'})
CREATE (p)-[:WATCHED {date: '2023-01-02'}]->(m)

In [ ]:
%%neo4j # list Persons
MATCH (p:Person)
RETURN p

p
"(_0:Person {age: 1, name: 'Alice'})"
"(_1:Person {age: 2, name: 'Bob'})"
"(_2:Person {age: 35, city: 'New York', name: 'Charlie'})"


In [ ]:
%%neo4j # returns attributes
MATCH (p:Person)
RETURN p.name, p.age

p.name,p.age
Alice,1
Bob,2
Charlie,35


In [ ]:
%%neo4j # returns Persons in a particular city
MATCH (p:Person)
WHERE p.city = 'New York'
RETURN p

p
"(_2:Person {age: 35, city: 'New York', name: 'Charlie'})"


In [ ]:
%%neo4j # returns Persons who watched a Movie
MATCH (p:Person)-[w:WATCHED]->(m:Movie)
WHERE m.title = 'Inception'
RETURN p,w,id(w)

p,w,id(w)
"(_2:Person {age: 35, city: 'New York', name: 'Charlie'})",(Charlie)-[:WATCHED {date: '2023-01-01'}]->(_3),1


In [ ]:
%%neo4j # complex query
MATCH (p:Person)
WHERE p.age > 30 AND p.city = 'New York'
RETURN p

p
"(_2:Person {age: 35, city: 'New York', name: 'Charlie'})"


In [ ]:
%%neo4j # Aggregation queries
MATCH (p:Person)
RETURN COUNT(p)

COUNT(p)
3


In [ ]:
%%neo4j # aggregation queries with rename
MATCH (p:Person)
RETURN p.city, COUNT(p) AS numberOfPeople

p.city,numberOfPeople
null,2
New York,1


In [ ]:
%%neo4j # Agregation
MATCH (p:Person)
RETURN AVG(p.age) AS averageAge

averageAge
12.666666666666666


In [ ]:
%%neo4j # find the shortest path between Alice and Bob
MATCH (p1:Person {name: 'Alice'}), (p2:Person {name: 'Bob'})
MATCH path = shortestPath((p1)-[*]-(p2))
RETURN path

path
(Alice)-[:KNOWS {}]->(Bob)


In [ ]:
%%neo4j # Using Pattern Matching
MATCH (p:Person)-[:WATCHED]->(m:Movie)
WHERE m.release_year > 2000
RETURN p.name, m.title

p.name,m.title
Charlie,Inception
Charlie,interstellar


In [ ]:
%%neo4j # Using WITH Clause for Intermediate Results
MATCH (p:Person)-[:WATCHED]->(m:Movie)
WITH p, COUNT(m) AS moviesWatched
WHERE moviesWatched >= 2
RETURN p.name, moviesWatched

p.name,moviesWatched
Charlie,2


In [ ]:
%%neo4j # Create an Index
CREATE INDEX FOR (p:Person) ON (p.name)

In [ ]:
%%neo4j # Create a Unique Constraint
CREATE CONSTRAINT FOR (p:Person) REQUIRE p.email IS UNIQUE

In [ ]:
%%neo4j
CREATE (a:Person {name: 'John 1234', age: 1, email:"abc@gmail.com"})
CREATE (b:Person {name: 'May 7890', age: 2, email:"abc@gmail.com"})

ClientError: [Schema.ConstraintValidationFailed] Node(9) already exists with label `Person` and property `email` = 'abc@gmail.com'

Backup and Restore

In [ ]:
!nj/bin/neo4j stop

Neo4j is not running.


In [ ]:
!nj/bin/neo4j-admin database info

Database name:                neo4j
Database in use:              false
Store format version:         record-aligned-1.1
Store format introduced in:   5.0.0
Last committed transaction id:25
Store needs recovery:         false

Database name:                system
Database in use:              false
Store format version:         record-aligned-1.1
Store format introduced in:   5.0.0
Last committed transaction id:41
Store needs recovery:         false


In [ ]:
!nj/bin/neo4j-admin database dump neo4j --to-path=/content/

2024-07-31 15:24:31.075+0000 INFO  [o.n.c.d.DumpCommand] Starting dump of database 'neo4j'
Done: 38 files, 258.0MiB processed.
2024-07-31 15:24:32.906+0000 INFO  [o.n.c.d.DumpCommand] Dump completed successfully


In [ ]:
!nj/bin/neo4j start

Directories in use:
home:         /content/nj
config:       /content/nj/conf
logs:         /content/nj/logs
plugins:      /content/nj/plugins
import:       /content/nj/import
data:         /content/nj/data
certificates: /content/nj/certificates
licenses:     /content/nj/licenses
run:          /content/nj/run
Starting Neo4j.
Started neo4j (pid:14643). It is available at http://localhost:7474
There may be a short delay until the server is ready.


In [ ]:
graph = Graph("bolt://localhost:7687")
graph

Graph('bolt://localhost:7687')

In [ ]:
%%neo4j
MATCH (m)
DETACH DELETE m

In [ ]:
%%neo4j
MATCH (n) RETURN n LIMIT 1

In [ ]:
!nj/bin/neo4j stop

Stopping Neo4j............ stopped.


In [ ]:
!nj/bin/neo4j-admin database load neo4j --from-path=/content/ --overwrite-destination=true

Done: 38 files, 258.0MiB processed.


In [ ]:
!nj/bin/neo4j start

Directories in use:
home:         /content/nj
config:       /content/nj/conf
logs:         /content/nj/logs
plugins:      /content/nj/plugins
import:       /content/nj/import
data:         /content/nj/data
certificates: /content/nj/certificates
licenses:     /content/nj/licenses
run:          /content/nj/run
Starting Neo4j.
Started neo4j (pid:14742). It is available at http://localhost:7474
There may be a short delay until the server is ready.


In [ ]:
graph = Graph("bolt://localhost:7687")
graph

Graph('bolt://localhost:7687')

In [ ]:
%%neo4j
MATCH (n) RETURN n

n
"(_0:Person {age: 1, name: 'Alice'})"
"(_1:Person {age: 2, name: 'Bob'})"
"(_2:Person {age: 35, city: 'New York', name: 'Charlie'})"
"(_3:Movie {release_year: 2010, title: 'Inception'})"
"(_4:Movie {release_year: 2014, title: 'interstellar'})"


Data Modification

In [ ]:
%%neo4j # Update Node Properties
MATCH (p:Person {name: 'Charlie'})
SET p.age = 36
RETURN p

p
"(_2:Person {age: 36, city: 'New York', name: 'Charlie'})"


In [ ]:
%%neo4j # Update Node Properties
MATCH (p:Person {name: 'Alice'})
SET p.age = 25
RETURN p

p
"(_0:Person {age: 25, name: 'Alice'})"


In [ ]:
%%neo4j # Update Node Properties
MATCH (p:Person {name: 'Bob'})
SET p.age = 30
RETURN p

p
"(_1:Person {age: 30, name: 'Bob'})"


Using Functions

In [ ]:
# %%neo4j # Delete a Node and Its Relationships
# MATCH (p:Person {name: 'Charlie'})
# DETACH DELETE p

In [ ]:
%%neo4j # String Functions
MATCH (p:Person)
RETURN p.name, toUpper(p.name) AS upperName

p.name,upperName
Alice,ALICE
Bob,BOB
Charlie,CHARLIE
John,JOHN
May,MAY
John 1234,JOHN 1234
May 7890,MAY 7890


In [ ]:
%%neo4j # Mathematical Functions
MATCH (p:Person)
RETURN p.name, p.age, p.age * 2 AS doubleAge

p.name,p.age,doubleAge
Alice,1,2
Bob,2,4
Charlie,35,70
John,1,2
May,2,4
John 1234,1,2
May 7890,2,4


In [ ]:
%%neo4j # Date Functions
MATCH (p:Person)-[r:WATCHED]->(m:Movie)
RETURN p.name, m.title, r.date, date(r.date).year AS watchYear

p.name,m.title,r.date,watchYear
Charlie,Inception,2023-01-01,2023
Charlie,interstellar,2023-01-02,2023


Working with Lists

In [ ]:
%%neo4j # Create and Access Lists
CREATE (p:Person {name: 'David', hobbies: ['reading', 'hiking', 'coding']})
WITH p
MATCH (p:Person {name: 'David'})
RETURN p.name, p.hobbies[0] AS firstHobby

p.name,firstHobby
David,reading


In [ ]:
%%neo4j # Filter Nodes Based on List Contents
MATCH (p:Person)
WHERE 'coding' IN p.hobbies
RETURN p.name, p.hobbies

p.name,p.hobbies
David,"['reading', 'hiking', 'coding']"


Subqueries and Nested Queries

In [ ]:
%%neo4j
MATCH (p:Person)
MATCH (p)-[:WATCHED]->(m:Movie)
WITH p, COUNT(m) AS moviesWatched
WHERE moviesWatched >= 2
RETURN p.name, moviesWatched

p.name,moviesWatched
Charlie,2


In [ ]:
%%neo4j # Using Subqueries
MATCH (p:Person)
WITH p, size([(p)-[:WATCHED]->(m:Movie) | m]) AS moviesWatched
WHERE moviesWatched >= 2
RETURN p.name, moviesWatched

p.name,moviesWatched
Charlie,2


In [ ]:
%%neo4j # Nested Queries, Using WITH Clause to Pass Variables
MATCH (p:Person)
WITH p
MATCH (p)-[:WATCHED]->(m:Movie)
WHERE m.title = 'Inception'
RETURN p.name

p.name
Charlie


Complex Exercise

In [ ]:
%%neo4j #
CREATE (p1:Person {name: 'Emma', age: 27}),
       (p2:Person {name: 'Liam', age: 31}),
       (p3:Person {name: 'Olivia', age: 29}),
       (c:Company {name: 'InnovateTech'}),
       (p1)-[:WORKS_AT]->(c),
       (p2)-[:WORKS_AT]->(c),
       (p3)-[:WORKS_AT]->(c)

In [ ]:
%%neo4j #
MATCH (p:Person)-[:WORKS_AT]->(c:Company {name: 'InnovateTech'})
RETURN p.name, p.age, c.name

p.name,p.age,c.name
Emma,27,InnovateTech
Liam,31,InnovateTech
Olivia,29,InnovateTech


In [ ]:
%%neo4j #
MATCH (p:Person {name: 'Emma'})
SET p.city = 'San Francisco'
RETURN p

p
"(_6:Person {age: 27, city: 'San Francisco', name: 'Emma'})"


In [ ]:
%%neo4j #
MATCH (p:Person {name: 'Liam'})
DETACH DELETE p

In [ ]:
%%neo4j #
MATCH (p:Person)
RETURN p

p
"(_0:Person {age: 25, name: 'Alice'})"
"(_1:Person {age: 30, name: 'Bob'})"
"(_2:Person {age: 36, city: 'New York', name: 'Charlie'})"
"(_5:Person {hobbies: ['reading', 'hiking', 'coding'], name: 'David'})"
"(_6:Person {age: 27, city: 'San Francisco', name: 'Emma'})"
"(_8:Person {age: 29, name: 'Olivia'})"


In [ ]:
%%neo4j # Find All Paths Between Two Nodes. Instead of 'Find Shortest Path Between Two Nodes'
MATCH path = (p1:Person {name: 'Alice'})-[*]-(p2:Person {name: 'Bob'})
RETURN path

path
(Alice)-[:KNOWS {}]->(Bob)


In [ ]:
%%neo4j # Find All Simple Paths with a Maximum Length
MATCH (p1:Person {name: 'Alice'}), (p2:Person {name: 'Bob'})
MATCH path = (p1)-[*..3]-(p2)
RETURN path

path
(Alice)-[:KNOWS {}]->(Bob)


In [ ]:
%%neo4j #
MATCH (p:Person)
WHERE p.age <= 27
SET p.status = 'Young'
RETURN p

p
"(_0:Person {age: 25, name: 'Alice', status: 'Young'})"
"(_6:Person {age: 27, city: 'San Francisco', name: 'Emma', status: 'Young'})"


In [ ]:
%%neo4j # Create a Node with a List Property
CREATE (p:Person {name: 'Eve', scores: [10, 20, 30, 40]})

In [ ]:
%%neo4j # Use REDUCE to Sum the Values in the List
MATCH (p:Person {name: 'Eve'})
RETURN p.name, REDUCE(total = 0, score IN p.scores | total + score) AS totalScore

p.name,totalScore
Eve,100


Map/Reduce queries

In [ ]:
%%neo4j # Create Person and Transaction Nodes
CREATE (p:Person {name: 'Frank'})
CREATE (t1:Transaction {amount: 100, date: '2023-01-01'})
CREATE (t2:Transaction {amount: 200, date: '2023-02-01'})
CREATE (t3:Transaction {amount: 300, date: '2023-03-01'})

In [ ]:
%%neo4j # Create Relationships Between Person and Transaction Nodes
MATCH (p:Person {name: 'Frank'}), (t1:Transaction {amount: 100, date: '2023-01-01'})
CREATE (p)-[:MADE]->(t1)
WITH p
MATCH (t2:Transaction {amount: 200, date: '2023-02-01'})
CREATE (p)-[:MADE]->(t2)
WITH p
MATCH (t3:Transaction {amount: 300, date: '2023-03-01'})
CREATE (p)-[:MADE]->(t3)

In [ ]:
%%neo4j # Use REDUCE to Sum the Amounts of Transactions
MATCH (p:Person {name: 'Frank'})-[:MADE]->(t:Transaction)
RETURN p.name, REDUCE(total = 0, transaction IN COLLECT(t) | total + transaction.amount) AS totalAmount

p.name,totalAmount
Frank,600


More Relations

In [ ]:
%%neo4j # Clear persons
MATCH (m)
DETACH DELETE m

In [ ]:
%%neo4j
CREATE (alice:Person {name: 'Alice', age: 30}),
       (alicia:Person {name: 'Alicia', age: 40}),
       (alice2:Person {name: 'alice', age: 50}),
       (bob:Person {name: 'Bob', age: 35}),
       (bobby:Person {name: 'Bobby', age: 38}),
       (charlie:Person {name: 'Charlie', age: 25}),
       (david:Person {name: 'David', age: 40}),
       (eve:Person {name: 'Eve', age: 28}),
       (frank:Person {name: 'Frank', age: 33}),
       (grace:Person {name: 'Grace', age: 29}),
       (alice)-[:FRIEND {since: date('2020-01-01')}]->(bob),
       (alice)-[:FRIEND {since: date('2021-02-01')}]->(charlie),
       (bob)-[:FRIEND {since: date('2020-03-01')}]->(david),
       (charlie)-[:FRIEND {since: date('2021-04-01')}]->(eve),
       (david)-[:FRIEND {since: date('2020-05-01')}]->(frank),
       (eve)-[:FRIEND {since: date('2022-06-01')}]->(grace),
       (alice)-[:MANAGES {since: date('2023-01-01')}]->(bob),
       (bob)-[:MANAGES {since: date('2023-02-01')}]->(charlie),
       (charlie)-[:MANAGES {since: date('2024-03-01')}]->(david),
       (david)-[:MANAGES {since: date('2024-04-01')}]->(eve),
       (eve)-[:MANAGES {since: date('2024-05-01')}]->(frank),
       (frank)-[:MANAGES {since: date('2024-06-01')}]->(grace)

In [ ]:
%%neo4j
MATCH (p:Person {name: 'Alice'})-[:FRIEND]->(friend)
RETURN friend.name AS friendName

friendName
Charlie
Bob


In [ ]:
%%neo4j
MATCH (manager:Person {name: 'Alice'})-[:MANAGES]->(employee)
RETURN employee.name AS employeeName

employeeName
Bob


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Alice'})-[:FRIEND*1..2]-(fof)
RETURN DISTINCT fof.name AS friendOfFriend

friendOfFriend
Charlie
Bob
David
Eve


In [ ]:
%%neo4j
MATCH (manager:Person {name: 'Alice'})-[:MANAGES*1..]-(employee)
RETURN DISTINCT employee.name AS employeeName

employeeName
Bob
Charlie
David
Eve
Frank
Grace


In [ ]:
%%neo4j
MATCH (manager:Person {name: 'Alice'})-[:MANAGES*1..]-(employee)
RETURN DISTINCT employee.name AS employeeName

employeeName
Bob
Charlie
David
Eve
Frank
Grace


In [ ]:
%%neo4j
MATCH (p1:Person {name: 'Alice'}), (p2:Person {name: 'Grace'})
MATCH path = shortestPath((p1)-[*]-(p2))
RETURN path

path
(Alice)-[:FRIEND {since: date('2021-02-01')}]->(Charlie)-[:FRIEND {since: date('2021-04-01')}]->(Eve)-[:FRIEND {since: date('2022-06-01')}]->(Grace)


In [ ]:
%%neo4j
MATCH path = (p1:Person {name: 'Alice'})-[:FRIEND|MANAGES*]-(p2:Person {name: 'Grace'})
RETURN path
LIMIT 5

path
(Alice)-[:FRIEND {since: date('2020-01-01')}]->(Bob)-[:FRIEND {since: date('2020-03-01')}]->(David)-[:FRIEND {since: date('2020-05-01')}]->(Frank)<-[:MANAGES {since: date('2024-05-01')}]-(Eve)<-[:FRIEND {since: date('2021-04-01')}]-(Charlie)<-[:FRIEND {since: date('2021-02-01')}]-(Alice)-[:MANAGES {since: date('2023-01-01')}]->(Bob)-[:MANAGES {since: date('2023-02-01')}]->(Charlie)-[:MANAGES {since: date('2024-03-01')}]->(David)-[:MANAGES {since: date('2024-04-01')}]->(Eve)-[:FRIEND {since: date('2022-06-01')}]->(Grace)
(Alice)-[:FRIEND {since: date('2020-01-01')}]->(Bob)-[:FRIEND {since: date('2020-03-01')}]->(David)-[:FRIEND {since: date('2020-05-01')}]->(Frank)<-[:MANAGES {since: date('2024-05-01')}]-(Eve)<-[:FRIEND {since: date('2021-04-01')}]-(Charlie)<-[:MANAGES {since: date('2023-02-01')}]-(Bob)<-[:MANAGES {since: date('2023-01-01')}]-(Alice)-[:FRIEND {since: date('2021-02-01')}]->(Charlie)-[:MANAGES {since: date('2024-03-01')}]->(David)-[:MANAGES {since: date('2024-04-01')}]->(Eve)-[:FRIEND {since: date('2022-06-01')}]->(Grace)
(Alice)-[:FRIEND {since: date('2020-01-01')}]->(Bob)-[:FRIEND {since: date('2020-03-01')}]->(David)-[:FRIEND {since: date('2020-05-01')}]->(Frank)<-[:MANAGES {since: date('2024-05-01')}]-(Eve)<-[:FRIEND {since: date('2021-04-01')}]-(Charlie)-[:MANAGES {since: date('2024-03-01')}]->(David)-[:MANAGES {since: date('2024-04-01')}]->(Eve)-[:FRIEND {since: date('2022-06-01')}]->(Grace)
(Alice)-[:FRIEND {since: date('2020-01-01')}]->(Bob)-[:FRIEND {since: date('2020-03-01')}]->(David)-[:FRIEND {since: date('2020-05-01')}]->(Frank)<-[:MANAGES {since: date('2024-05-01')}]-(Eve)-[:FRIEND {since: date('2022-06-01')}]->(Grace)
(Alice)-[:FRIEND {since: date('2020-01-01')}]->(Bob)-[:FRIEND {since: date('2020-03-01')}]->(David)-[:FRIEND {since: date('2020-05-01')}]->(Frank)<-[:MANAGES {since: date('2024-05-01')}]-(Eve)<-[:MANAGES {since: date('2024-04-01')}]-(David)<-[:MANAGES {since: date('2024-03-01')}]-(Charlie)<-[:FRIEND {since: date('2021-02-01')}]-(Alice)-[:MANAGES {since: date('2023-01-01')}]->(Bob)-[:MANAGES {since: date('2023-02-01')}]->(Charlie)-[:FRIEND {since: date('2021-04-01')}]->(Eve)-[:FRIEND {since: date('2022-06-01')}]->(Grace)
(Alice)-[:FRIEND {since: date('2020-01-01')}]->(Bob)-[:FRIEND {since: date('2020-03-01')}]->(David)-[:FRIEND {since: date('2020-05-01')}]->(Frank)<-[:MANAGES {since: date('2024-05-01')}]-(Eve)<-[:MANAGES {since: date('2024-04-01')}]-(David)<-[:MANAGES {since: date('2024-03-01')}]-(Charlie)-[:FRIEND {since: date('2021-04-01')}]->(Eve)-[:FRIEND {since: date('2022-06-01')}]->(Grace)
(Alice)-[:FRIEND {since: date('2020-01-01')}]->(Bob)-[:FRIEND {since: date('2020-03-01')}]->(David)-[:FRIEND {since: date('2020-05-01')}]->(Frank)<-[:MANAGES {since: date('2024-05-01')}]-(Eve)<-[:MANAGES {since: date('2024-04-01')}]-(David)<-[:MANAGES {since: date('2024-03-01')}]-(Charlie)<-[:MANAGES {since: date('2023-02-01')}]-(Bob)<-[:MANAGES {since: date('2023-01-01')}]-(Alice)-[:FRIEND {since: date('2021-02-01')}]->(Charlie)-[:FRIEND {since: date('2021-04-01')}]->(Eve)-[:FRIEND {since: date('2022-06-01')}]->(Grace)
(Alice)-[:FRIEND {since: date('2020-01-01')}]->(Bob)-[:FRIEND {since: date('2020-03-01')}]->(David)-[:FRIEND {since: date('2020-05-01')}]->(Frank)-[:MANAGES {since: date('2024-06-01')}]->(Grace)
(Alice)-[:FRIEND {since: date('2020-01-01')}]->(Bob)-[:FRIEND {since: date('2020-03-01')}]->(David)<-[:MANAGES {since: date('2024-03-01')}]-(Charlie)<-[:FRIEND {since: date('2021-02-01')}]-(Alice)-[:MANAGES {since: date('2023-01-01')}]->(Bob)-[:MANAGES {since: date('2023-02-01')}]->(Charlie)-[:FRIEND {since: date('2021-04-01')}]->(Eve)-[:FRIEND {since: date('2022-06-01')}]->(Grace)
(Alice)-[:FRIEND {since: date('2020-01-01')}]->(Bob)-[:FRIEND {since: date('2020-03-01')}]->(David)<-[:MANAGES {since: date('2024-03-01')}]-(Charlie)<-[:FRIEND {since: date('2021-02-01')}]-(Alice)-[:MANAGES {since: date('2023-01-01')}]->(Bob)-[:MANAGES {since: date('2023-02-01')}]->(Charlie)-[:FRIEND {si

In [ ]:
%%neo4j
MATCH (p:Person)-[:FRIEND]-(friend)
RETURN p.name AS personName, COUNT(friend) AS numberOfFriends
ORDER BY numberOfFriends DESC
LIMIT 1

personName,numberOfFriends
Alice,2


In [ ]:
%%neo4j
MATCH (manager:Person {name: 'Alice'})-[:MANAGES]->(employee)-[:FRIEND]->(friend)
RETURN employee.name AS employeeName, friend.name AS friendName

employeeName,friendName
Bob,David


In [ ]:
%%neo4j
MATCH (p:Person)-[r:FRIEND]->(friend)
WHERE r.since.year = date().year - 2
RETURN p.name, friend.name, r.since

p.name,friend.name,r.since
Eve,Grace,date('2022-06-01')


In [ ]:
%%neo4j
MATCH (manager:Person {name: 'Alice'})-[r:MANAGES]->(employee)
RETURN manager.name, employee.name, duration.between(r.since, date()).months AS monthsManaged

manager.name,employee.name,monthsManaged
Alice,Bob,19


In [ ]:
%%neo4j
MATCH (p:Person)
WITH collect(p) AS persons
FOREACH (p IN persons | SET p.status = 'Updated')
RETURN persons

persons
"[(_13:Person {age: 30, name: 'Alice', status: 'Updated'}), (_14:Person {age: 40, name: 'Alicia', status: 'Updated'}), (_15:Person {age: 50, name: 'alice', status: 'Updated'}), (_16:Person {age: 35, name: 'Bob', status: 'Updated'}), (_17:Person {age: 38, name: 'Bobby', status: 'Updated'}), (_18:Person {age: 25, name: 'Charlie', status: 'Updated'}), (_19:Person {age: 40, name: 'David', status: 'Updated'}), (_20:Person {age: 28, name: 'Eve', status: 'Updated'}), (_21:Person {age: 33, name: 'Frank', status: 'Updated'}), (_22:Person {age: 29, name: 'Grace', status: 'Updated'})]"


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Alice'}), (friends:Person)
WHERE friends.name IN ['Bob', 'Charlie']
WITH p, collect(friends) AS friendsList
FOREACH (friend IN friendsList |
  MERGE (p)-[:FRIEND {since: date()}]->(friend)
)
RETURN p, friendsList

p,friendsList
"(_13:Person {age: 30, name: 'Alice', status: 'Updated'})","[(_16:Person {age: 35, name: 'Bob', status: 'Updated'}), (_18:Person {age: 25, name: 'Charlie', status: 'Updated'})]"


In [ ]:
%%neo4j
UNWIND ['Alice', 'Bob', 'Charlie'] AS name
CREATE (p:Person {name: name})
WITH p
MATCH (m:Movie {title: 'Inception'})
CREATE (p)-[:WATCHED]->(m)
RETURN p, m

In [ ]:
%%neo4j
MATCH (p:Person)
WITH collect(p.name) AS names
UNWIND names AS name
WITH name, size(name) AS nameLength
WHERE nameLength > 3
RETURN name

name
Alice
Alice
Alicia
Bobby
Charlie
Charlie
David
Frank
Grace
alice


In [ ]:
%%neo4j
MATCH (p:Person)
SET p.status = CASE
  WHEN p.age < 20 THEN 'Teen'
  WHEN p.age < 30 THEN 'Young Adult'
  ELSE 'Adult'
END
RETURN p

p
"(_0:Person {name: 'Charlie', status: 'Adult'})"
"(_13:Person {age: 30, name: 'Alice', status: 'Adult'})"
"(_14:Person {age: 40, name: 'Alicia', status: 'Adult'})"
"(_15:Person {age: 50, name: 'alice', status: 'Adult'})"
"(_16:Person {age: 35, name: 'Bob', status: 'Adult'})"
"(_17:Person {age: 38, name: 'Bobby', status: 'Adult'})"
"(_18:Person {age: 25, name: 'Charlie', status: 'Young Adult'})"
"(_19:Person {age: 40, name: 'David', status: 'Adult'})"
"(_20:Person {age: 28, name: 'Eve', status: 'Young Adult'})"
"(_21:Person {age: 33, name: 'Frank', status: 'Adult'})"


In [ ]:
%%neo4j
MERGE (p:Person {name: 'Alice'})
ON CREATE SET p.created = timestamp()
ON MATCH SET p.lastSeen = timestamp()
RETURN p

p
"(_13:Person {age: 30, lastSeen: 1722879247369, name: 'Alice', status: 'Adult'})"
"(_23:Person {lastSeen: 1722879247369, name: 'Alice', status: 'Adult'})"


In [ ]:
%%neo4j
MATCH (p1:Person {name: 'Alice'}), (p2:Person {name: 'Bob'})
MERGE (p1)-[r:FRIEND]->(p2)
ON CREATE SET r.since = date()
RETURN p1, p2, r

p1,p2,r
"(_13:Person {age: 30, lastSeen: 1722879247369, name: 'Alice', status: 'Adult'})","(_16:Person {age: 35, name: 'Bob', status: 'Adult'})",(Alice)-[:FRIEND {since: date('2024-08-05')}]->(Bob)
"(_13:Person {age: 30, lastSeen: 1722879247369, name: 'Alice', status: 'Adult'})","(_16:Person {age: 35, name: 'Bob', status: 'Adult'})",(Alice)-[:FRIEND {since: date('2020-01-01')}]->(Bob)
"(_13:Person {age: 30, lastSeen: 1722879247369, name: 'Alice', status: 'Adult'})","(_24:Person {name: 'Bob', status: 'Adult'})",(Alice)-[:FRIEND {since: date('2024-08-05')}]->(Bob)
"(_23:Person {lastSeen: 1722879247369, name: 'Alice', status: 'Adult'})","(_16:Person {age: 35, name: 'Bob', status: 'Adult'})",(Alice)-[:FRIEND {since: date('2024-08-05')}]->(Bob)
"(_23:Person {lastSeen: 1722879247369, name: 'Alice', status: 'Adult'})","(_24:Person {name: 'Bob', status: 'Adult'})",(Alice)-[:FRIEND {since: date('2024-08-05')}]->(Bob)


# Analysing  APOC functions

In [ ]:
%%neo4j
CALL apoc.help('apoc.text') YIELD name, text
RETURN name, text

name,text
apoc.text.phoneticDelta,Returns the US_ENGLISH soundex character difference between the two given `STRING` values.
apoc.text.base64Decode,Decodes the given Base64 encoded `STRING`.
apoc.text.base64Encode,Encodes the given `STRING` with Base64.
apoc.text.base64UrlDecode,Decodes the given Base64 encoded URL.
apoc.text.base64UrlEncode,Encodes the given URL with Base64.
apoc.text.byteCount,Returns the size of the given `STRING` in bytes.
apoc.text.bytes,Returns the given `STRING` as bytes.
apoc.text.camelCase,Converts the given `STRING` to camel case.
apoc.text.capitalize,Capitalizes the first letter of the given `STRING`.
apoc.text.capitalizeAll,Capitalizes the first letter of every word in the given `STRING`.


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Alice'})
WITH apoc.text.base64Encode(p.name) AS encodedName
RETURN encodedName, apoc.text.base64Decode(encodedName) AS decodedName

encodedName,decodedName
QWxpY2U=,Alice
QWxpY2U=,Alice


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Grace'})
RETURN p.status, apoc.text.camelCase(p.status) AS camelCaseStatus

p.status,camelCaseStatus
Young Adult,youngAdult


In [ ]:
%%neo4j
MATCH (p:Person)
RETURN p.status, apoc.text.capitalize(p.status) AS capitalizedName
LIMIT 1

p.status,capitalizedName
Adult,Adult


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Eve'})
RETURN p.name, apoc.text.charAt(p.name, 1) AS charAtIndex1

p.name,charAtIndex1
Eve,118


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Frank'})
RETURN apoc.text.clean(p.name) AS cleanedName

cleanedName
frank


In [ ]:
%%neo4j
MATCH (p1:Person {name: 'Alice'}), (p2:Person {name: 'alice'})
RETURN apoc.text.compareCleaned(p1.name, p2.name) AS compareCleaned

compareCleaned
true
true


In [ ]:
%%neo4j
MATCH (p1:Person {name: 'Alice'}), (p2:Person {name: 'Alicia'})
RETURN apoc.text.distance(p1.name, p2.name) AS levenshteinDistance
LIMIT 1

levenshteinDistance
2


In [ ]:
%%neo4j
MATCH (p1:Person {name: 'Alice'}), (p2:Person {name: 'Alicia'})
RETURN apoc.text.levenshteinSimilarity(p1.name, p2.name) AS levenshteinSimilarity
LIMIT 1

levenshteinSimilarity
0.6666666666666666
0.6666666666666666


In [ ]:
%%neo4j
MATCH (p1:Person {name: 'Frank'}), (p2:Person {name: 'Grace'})
RETURN apoc.text.doubleMetaphone(p1.name) AS doubleMetaphone1, apoc.text.doubleMetaphone(p2.name)  AS doubleMetaphone2

doubleMetaphone1,doubleMetaphone2
FRNK,KRS


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Grace'})
RETURN apoc.text.format("Name: %s, Age: %d", [p.name, p.age]) AS formattedString

formattedString
"Name: Grace, Age: 29"


In [ ]:
%%neo4j
MATCH (p1:Person {name: 'Alice'}), (p2:Person {name: 'Alicia'})
RETURN apoc.text.fuzzyMatch(p1.name, p2.name) AS fuzzyMatch

fuzzyMatch
true
true


In [ ]:
%%neo4j
MATCH (p1:Person {name: 'Bobyb'}), (p2:Person {name: 'Bobby'})
RETURN apoc.text.hammingDistance(p1.name, p2.name) AS hammingDistance

In [ ]:
%%neo4j
MATCH (p:Person {name: 'Charlie'})
RETURN apoc.text.indexOf(p.name, 'ar') AS indexOfSubstring

indexOfSubstring
2
2


In [ ]:
%%neo4j
MATCH (p:Person)
WITH collect(p.name) AS names
RETURN apoc.text.join(names, ', ') AS joinedNames

joinedNames
"Alice, Alice, Alicia, Bob, Bob, Bobby, Charlie, Charlie, David, Eve, Frank, Grace, alice"


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Eve'})
RETURN apoc.text.lpad(p.name, 10, ' ') AS leftPaddedName

leftPaddedName
Eve


In [ ]:
%%neo4j
MATCH (p:Person {name: 'David'})
RETURN apoc.text.phonetic(p.name) AS phoneticEncoding

phoneticEncoding
D130


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Charlie'})
RETURN apoc.text.regexGroups(p.name, '(C)(h)(a)(r)(l)(i)(e)') AS regexGroups
LIMIT 1

regexGroups
"[['Charlie', 'C', 'h', 'a', 'r', 'l', 'i', 'e']]"


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Frank'})
RETURN apoc.text.replace(p.name, 'a', '@') AS replacedName
LIMIT 1

replacedName
Fr@nk


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Grace'})
RETURN apoc.text.split(p.name, 'a') AS splitName
LIMIT 1

splitName
"['Gr', 'ce']"


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Eve'})
RETURN apoc.text.slug(p.status, '-') AS slugStatus

slugStatus
Young-Adult


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Alice'})
RETURN apoc.text.snakeCase(p.status) AS snakeCaseStatus

snakeCaseStatus
adult
adult


In [ ]:
%%neo4j
MATCH (p1:Person {name: 'Alice'}), (p2:Person {name: 'Alicia'})
RETURN apoc.text.sorensenDiceSimilarity(p1.name, p2.name, 'en') AS sorensenDiceSimilarity

sorensenDiceSimilarity
0.6666666666666666
0.6666666666666666


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Grace'})
RETURN apoc.text.swapCase(p.name) AS swappedCaseName

swappedCaseName
gRACE


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Frank'})
RETURN apoc.text.toUpperCase(p.name) AS upperCaseName

upperCaseName
FRANK


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Eve'})
RETURN apoc.text.upperCamelCase(p.status) AS upperCamelCaseStatus

upperCamelCaseStatus
YoungAdult


In [ ]:
%%neo4j
WITH 'https%3A%2F%2Fexample.com%2Fpath%3Fquery%3Dvalue' AS encodedUrl
RETURN apoc.text.urldecode(encodedUrl) AS decodedUrl,
       apoc.text.urlencode('https://example.com/path?query=value') AS encodedUrl

decodedUrl,encodedUrl
https://example.com/path?query=value,https%3A%2F%2Fexample.com%2Fpath%3Fquery%3Dvalue


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Charlie'})
RETURN apoc.text.byteCount(p.name) AS byteCount

byteCount
7
7


In [ ]:
%%neo4j
MATCH (p1:Person {name: 'Alice'}), (p2:Person {name: 'Alicia'})
RETURN apoc.text.jaroWinklerDistance(p1.name, p2.name) AS jaroWinklerDistance

jaroWinklerDistance
0.10666666666666658
0.10666666666666658


In [ ]:
%%neo4j
MATCH (p:Person {name: 'David'})
RETURN apoc.text.regexGroups(p.name, '(D)(a)(v)(i)(d)') AS regexGroups

regexGroups
"[['David', 'D', 'a', 'v', 'i', 'd']]"


In [ ]:
%%neo4j
RETURN apoc.text.repeat('Neo4j', 3) AS repeatedString

repeatedString
Neo4jNeo4jNeo4j


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Eve'})
RETURN apoc.text.rpad(p.name, 10, '>') AS rightPaddedName, apoc.text.lpad(p.name, 10, '<') AS leftPaddedName

rightPaddedName,leftPaddedName
Eve>>>>>>>,<<<<<<<Eve


In [ ]:
%%neo4j
MATCH (p:Person {name: 'Alice'})
RETURN apoc.text.toCypher(p.name) AS cypherString

cypherString
'Alice'
'Alice'


## Managing Candidate Resumes

In [ ]:
%%neo4j #
CREATE (c:Candidate {
  full_name: "SHAIK SAHUL HAMEED AZEEM.A",
  email: "azeembc16@gmail.com",
  summary: null,
  cv_path: "1002.pdf"
})

In [ ]:
%%neo4j # Create Soft Skills Nodes and Relationships
MATCH (c:Candidate {email: "azeembc16@gmail.com"})
WITH c
UNWIND [
  "Excellent communicator",
  "Problem solving"
] AS skill
CREATE (s:SoftSkill {description: skill})
CREATE (c)-[:HAS_SOFT_SKILL]->(s)

In [ ]:
%%neo4j # Create Hard Skills Nodes and Relationships
MATCH (c:Candidate {email: "azeembc16@gmail.com"})
WITH c
UNWIND [
  "SAP PM & MM",
  "Functional location",
  "Project scheduling"
] AS skill
CREATE (h:HardSkill {description: skill})
CREATE (c)-[:HAS_HARD_SKILL]->(h)

In [ ]:
%%neo4j # Create Education Nodes and Relationships
MATCH (c:Candidate {email: "azeembc16@gmail.com"})
WITH c
UNWIND [
  {course: "B.E / B.Tech - Mechanical Engineering", institute: "Anna University", date: "2007"},
  {course: "M.B.A (Pursuing) – Operations Management", institute: "Sikkim Manipal University", date: null}
] AS edu
CREATE (e:Education {course: edu.course, institute: edu.institute, date: edu.date})
CREATE (c)-[:HAS_EDUCATION]->(e)

In [ ]:
%%neo4j # Create Work Experience Nodes and Relationships
MATCH (c:Candidate {email: "azeembc16@gmail.com"})
WITH c
UNWIND [
  {
    title: "MAINTENANCE AREA PLANNER",
    info: "Resources arrangement, PM schedules, daily meetings",
    timeframe: "Since April 2013"
  },
  {
    title: "TURNAROUND (TA) PLANNER",
    info: "Prepare requirements, review PMNs, develop tender cost summaries",
    timeframe: "March 2012 – March 2013"
  },
  {
    title: "PLANNER/SCHEDULER",
    info: "Project master schedule, cost control, technical support",
    timeframe: "SEP 2009 - Dec 2011"
  }
] AS work
CREATE (w:WorkExperience {
  title: work.title,
  info: work.info,
  timeframe: work.timeframe
})
CREATE (c)-[:HAS_WORK_EXPERIENCE]->(w)

Query the Database

In [ ]:
%%neo4j #
MATCH (c:Candidate {email: "azeembc16@gmail.com"})
RETURN c

c
"(_5:Candidate {cv_path: '1002.pdf', email: 'azeembc16@gmail.com', full_name: 'SHAIK SAHUL HAMEED AZEEM.A'})"


In [ ]:
%%neo4j #
MATCH (c:Candidate {email: "azeembc16@gmail.com"})-[:HAS_SOFT_SKILL]->(s:SoftSkill)
RETURN s.description as soft_skill

soft_skill
Excellent communicator
Problem solving


In [ ]:
%%neo4j #
MATCH (c:Candidate {email: "azeembc16@gmail.com"})-[:HAS_HARD_SKILL]->(h:HardSkill)
RETURN h.description as hard_skill

hard_skill
SAP PM & MM
Functional location
Project scheduling


In [ ]:
%%neo4j #
MATCH (c:Candidate {email: "azeembc16@gmail.com"})-[:HAS_EDUCATION]->(e:Education)
RETURN e.course, e.institute, e.date

e.course,e.institute,e.date
B.E / B.Tech - Mechanical Engineering,Anna University,2007
M.B.A (Pursuing) – Operations Management,Sikkim Manipal University,null


In [ ]:
%%neo4j #
MATCH (c:Candidate {email: "azeembc16@gmail.com"})-[:HAS_WORK_EXPERIENCE]->(w:WorkExperience)
RETURN w.title, w.info, w.timeframe

w.title,w.info,w.timeframe
MAINTENANCE AREA PLANNER,"Resources arrangement, PM schedules, daily meetings",Since April 2013
TURNAROUND (TA) PLANNER,"Prepare requirements, review PMNs, develop tender cost summaries",March 2012 – March 2013
PLANNER/SCHEDULER,"Project master schedule, cost control, technical support",SEP 2009 - Dec 2011


In [ ]:
%%neo4j # Add a summary into the candidate
MATCH (c:Candidate {email: "azeembc16@gmail.com"})
SET c.summary = "Experienced Mechanical Engineer with extensive background in maintenance planning and project management."
RETURN c

c
"(_5:Candidate {cv_path: '1002.pdf', email: 'azeembc16@gmail.com', full_name: 'SHAIK SAHUL HAMEED AZEEM.A', summary: 'Experienced Mechanical Engineer with extensive background in maintenance planning and project management.'})"


In [ ]:
%%neo4j # Add a New Soft Skill
MATCH (c:Candidate {email: "azeembc16@gmail.com"})
CREATE (s:SoftSkill {description: "Strong leadership"})
CREATE (c)-[:HAS_SOFT_SKILL]->(s)
RETURN s

s
(_24:SoftSkill {description: 'Strong leadership'})


In [ ]:
%%neo4j # Add a New Work Experience
MATCH (c:Candidate {email: "azeembc16@gmail.com"})
CREATE (w:WorkExperience {
  title: "Senior Mechanical Engineer",
  info: "Leading a team of engineers",
  timeframe: "Jan 2020 - Present"
})
CREATE (c)-[:HAS_WORK_EXPERIENCE]->(w)
RETURN w

w
"(_25:WorkExperience {info: 'Leading a team of engineers', timeframe: 'Jan 2020 - Present', title: 'Senior Mechanical Engineer'})"


In [ ]:
%%neo4j # Delete a Soft Skill
MATCH (c:Candidate {email: "azeembc16@gmail.com"})-[:HAS_SOFT_SKILL]->(s:SoftSkill {description: "Problem solving"})
DETACH DELETE s

In [ ]:
%%neo4j # Delete a Work Experience
MATCH (c:Candidate {email: "azeembc16@gmail.com"})-[:HAS_WORK_EXPERIENCE]->(w:WorkExperience {title: "Junior Production Planner"})
DETACH DELETE w

In [ ]:
%%neo4j #
MATCH (c:Candidate {email: "azeembc16@gmail.com"})
OPTIONAL MATCH (c)-[:HAS_SOFT_SKILL]->(s:SoftSkill)
OPTIONAL MATCH (c)-[:HAS_HARD_SKILL]->(h:HardSkill)
OPTIONAL MATCH (c)-[:HAS_EDUCATION]->(e:Education)
OPTIONAL MATCH (c)-[:HAS_WORK_EXPERIENCE]->(w:WorkExperience)
RETURN c AS Candidate,
       collect(s) AS SoftSkills,
       collect(h) AS HardSkills,
       collect(e) AS Education,
       collect(w) AS WorkExperience

Candidate                                                                                                                                                                                                                         | SoftSkills                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       | HardSkills                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [ ]:
%%neo4j # Query to Show Unique Relations of Candidate
MATCH (c:Candidate {email: "azeembc16@gmail.com"})
OPTIONAL MATCH (c)-[:HAS_SOFT_SKILL]->(s:SoftSkill)
OPTIONAL MATCH (c)-[:HAS_HARD_SKILL]->(h:HardSkill)
OPTIONAL MATCH (c)-[:HAS_EDUCATION]->(e:Education)
OPTIONAL MATCH (c)-[:HAS_WORK_EXPERIENCE]->(w:WorkExperience)
RETURN
  c.full_name AS CandidateName,
  c.email AS Email,
  COLLECT(DISTINCT s.description) AS SoftSkills,
  COLLECT(DISTINCT h.description) AS HardSkills,
  COLLECT(DISTINCT {course: e.course, institute: e.institute, date: e.date}) AS Education,
  COLLECT(DISTINCT {title: w.title, info: w.info, timeframe: w.timeframe}) AS WorkExperience

CandidateName,Email,SoftSkills,HardSkills,Education,WorkExperience
SHAIK SAHUL HAMEED AZEEM.A,azeembc16@gmail.com,"['Strong leadership', 'Excellent communicator']","['Project scheduling', 'Functional location', 'SAP PM & MM']","[{course: 'M.B.A (Pursuing) \u2013 Operations Management', institute: 'Sikkim Manipal University', date: null}, {course: 'B.E / B.Tech - Mechanical Engineering', institute: 'Anna University', date: '2007'}]","[{title: 'Senior Mechanical Engineer', timeframe: 'Jan 2020 - Present', info: 'Leading a team of engineers'}, {title: 'PLANNER/SCHEDULER', timeframe: 'SEP 2009 - Dec 2011', info: 'Project master schedule, cost control, technical support'}, {title: 'TURNAROUND (TA) PLANNER', timeframe: 'March 2012 \u2013 March 2013', info: 'Prepare requirements, review PMNs, develop tender cost summaries'}, {title: 'MAINTENANCE AREA PLANNER', timeframe: 'Since April 2013', info: 'Resources arrangement, PM schedules, daily meetings'}]"
